# DistilBERT with Optuna Hyperparameter Search
#Finding the best hyperparameters automatically using Optuna, then retraining with them.

**Hyperparams:**
- `lr_base`: transformer body learning rate
- `lr_head`: classifier head learning rate
- `label_smoothing`:  cross-entropy label smoothing
- `warmup_ratio`:  fraction of steps used for warmup
- `num_frozen_layers`: how many bottom DistilBERT layers to freeze (0–3)


In [2]:
!pip install optuna transformers torch scikit-learn pandas numpy matplotlib seaborn --quiet

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, copy
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from torch.optim import AdamW

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  #suppressing per-step logging

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Using device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
VRAM: 8.5 GB


In [ ]:
import os

# Fixed settings
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 256
BATCH_SIZE = 64
CLASSES    = ['Anxiety', 'Depression', 'SuicideWatch']

# Optuna search settings
N_TRIALS        = 20   # number of hyperparameter combinations to try
TRIAL_EPOCHS    = 3    # epochs per trial (keep short for speed)
FINAL_EPOCHS    = 4    # epochs for final retrain with best params

# All outputs go here
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}/')

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

# Loading the data
train_df = pd.read_csv('../data/train_bert.csv')
test_df  = pd.read_csv('../data/test_bert.csv')

le = LabelEncoder()
le.fit(CLASSES)
train_df['label_enc'] = le.transform(train_df['label'])
test_df['label_enc']  = le.transform(test_df['label'])

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [6]:
class RedditDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def make_optimizer(model, lr_base, lr_head):
    no_decay = ['bias', 'LayerNorm.weight']
    return AdamW([
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and 'classifier' not in n],
         'lr': lr_base, 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and 'classifier' not in n],
         'lr': lr_base, 'weight_decay': 0.0},
        {'params': [p for n, p in model.named_parameters() if 'classifier' in n],
         'lr': lr_head, 'weight_decay': 0.01},
    ])


def train_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    preds_dec  = le.inverse_transform(all_preds)
    labels_dec = le.inverse_transform(all_labels)
    macro_f1 = f1_score(labels_dec, preds_dec, average='macro')
    acc      = accuracy_score(labels_dec, preds_dec)
    auc      = roc_auc_score(all_labels, np.array(all_probs), multi_class='ovr', average='macro')
    return macro_f1, acc, auc, preds_dec, labels_dec


print('Dataset class and helper functions defined.')

Dataset class and helper functions defined.


In [8]:
#Optuna objective function:
#Each trial picks a set of hyperparameters, trains on fold 1 only (fast) and returns the best val Macro F1 achieved across TRIAL_EPOCHS epochs.

texts  = train_df['bert_text'].values
labels = train_df['label_enc'].values

#Uinge fold 1 only for search (stratified split)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, val_idx = next(iter(skf.split(texts, labels)))

def objective(trial):
    #Hyperparameter search space 
    lr_base          = trial.suggest_float('lr_base',          5e-6,  5e-5,  log=True)
    lr_head          = trial.suggest_float('lr_head',          1e-5,  1e-4,  log=True)
    label_smoothing  = trial.suggest_float('label_smoothing',  0.0,   0.15)
    warmup_ratio     = trial.suggest_float('warmup_ratio',     0.05,  0.20)
    num_frozen       = trial.suggest_int(  'num_frozen_layers', 0,    3)

    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    train_dataset = RedditDataset(texts[tr_idx], labels[tr_idx], tokenizer, MAX_LEN)
    val_dataset   = RedditDataset(texts[val_idx], labels[val_idx], tokenizer, MAX_LEN)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3
    ).to(device)

    #Freezing bottom N layers
    for layer in model.distilbert.transformer.layer[:num_frozen]:
        for param in layer.parameters():
            param.requires_grad = False

    optimizer   = make_optimizer(model, lr_base, lr_head)
    total_steps = len(train_loader) * TRIAL_EPOCHS
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(warmup_ratio * total_steps),
        num_training_steps=total_steps
    )

    best_f1 = 0.0
    for epoch in range(TRIAL_EPOCHS):
        train_epoch(model, train_loader, optimizer, scheduler, loss_fn)
        f1, _, _, _, _ = evaluate(model, val_loader)
        if f1 > best_f1:
            best_f1 = f1
        #Optuna pruning
        trial.report(f1, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

    del model
    torch.cuda.empty_cache()
    return best_f1


print(f'Objective defined. Running {N_TRIALS} trials x {TRIAL_EPOCHS} epochs each...')


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n===== Optuna Search Complete =====')
print(f'Best Trial Val Macro F1 : {study.best_value:.4f}')
print(f'Best Hyperparameters:')
for k, v in study.best_params.items():
    print(f'  {k:<22} = {v}')

Objective defined. Running 20 trials x 3 epochs each...


  0%|          | 0/20 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



===== Optuna Search Complete =====
Best Trial Val Macro F1 : 0.7525
Best Hyperparameters:
  lr_base                = 3.614035885256077e-05
  lr_head                = 2.2483260719902293e-05
  label_smoothing        = 0.04550552211671318
  warmup_ratio           = 0.0625973441901766
  num_frozen_layers      = 0


In [ ]:
# Visualising Optuna results
trial_nums  = [t.number for t in study.trials if t.value is not None]
trial_f1s   = [t.value  for t in study.trials if t.value is not None]
best_so_far = np.maximum.accumulate(trial_f1s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trial scores
axes[0].bar(trial_nums, trial_f1s, color='steelblue', alpha=0.7)
axes[0].plot(trial_nums, best_so_far, color='red', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Val Macro F1')
axes[0].set_title('Optuna Trial Scores')
axes[0].legend()

# Hyperparameter importances
importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()), color='coral')
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importances')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/optuna_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {RESULTS_DIR}/optuna_results.png')

In [10]:
#Retraining with best hyperparameters — full 5-fold CV 
best = study.best_params
print('Retraining with best params:')
for k, v in best.items():
    print(f'  {k} = {v}')

loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=best['label_smoothing'])
skf_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1, cv_acc, cv_auc = [], [], []

for fold, (tr_idx, val_idx) in enumerate(skf_cv.split(texts, labels)):
    print(f'\n--- Fold {fold+1}/5 ---')

    train_dataset = RedditDataset(texts[tr_idx],  labels[tr_idx],  tokenizer, MAX_LEN)
    val_dataset   = RedditDataset(texts[val_idx], labels[val_idx], tokenizer, MAX_LEN)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3
    ).to(device)

    for layer in model.distilbert.transformer.layer[:best['num_frozen_layers']]:
        for param in layer.parameters():
            param.requires_grad = False

    optimizer   = make_optimizer(model, best['lr_base'], best['lr_head'])
    total_steps = len(train_loader) * FINAL_EPOCHS
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(best['warmup_ratio'] * total_steps),
        num_training_steps=total_steps
    )

    best_f1, best_acc, best_auc = 0.0, 0.0, 0.0
    for epoch in range(FINAL_EPOCHS):
        train_epoch(model, train_loader, optimizer, scheduler, loss_fn)
        f1, acc, auc, _, _ = evaluate(model, val_loader)
        print(f'  Epoch {epoch+1} — Macro F1: {f1:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f}')
        if f1 > best_f1:
            best_f1, best_acc, best_auc = f1, acc, auc

    cv_f1.append(best_f1)
    cv_acc.append(best_acc)
    cv_auc.append(best_auc)
    print(f'  Fold {fold+1} best — F1: {best_f1:.4f} | Acc: {best_acc:.4f} | AUC: {best_auc:.4f}')

    del model
    torch.cuda.empty_cache()

print(f'\n===== CV Summary (Optuna params) =====')
print(f'Mean Macro F1 : {np.mean(cv_f1):.4f} +/- {np.std(cv_f1):.4f}')
print(f'Mean Acc      : {np.mean(cv_acc):.4f} +/- {np.std(cv_acc):.4f}')
print(f'Mean AUC-ROC  : {np.mean(cv_auc):.4f} +/- {np.std(cv_auc):.4f}')

Retraining with best params:
  lr_base = 3.614035885256077e-05
  lr_head = 2.2483260719902293e-05
  label_smoothing = 0.04550552211671318
  warmup_ratio = 0.0625973441901766
  num_frozen_layers = 0

--- Fold 1/5 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 — Macro F1: 0.7320 | Acc: 0.7339 | AUC: 0.8807
  Epoch 2 — Macro F1: 0.7430 | Acc: 0.7505 | AUC: 0.8958
  Epoch 3 — Macro F1: 0.7469 | Acc: 0.7469 | AUC: 0.8957
  Epoch 4 — Macro F1: 0.7501 | Acc: 0.7510 | AUC: 0.8954
  Fold 1 best — F1: 0.7501 | Acc: 0.7510 | AUC: 0.8954

--- Fold 2/5 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 — Macro F1: 0.7308 | Acc: 0.7339 | AUC: 0.8907
  Epoch 2 — Macro F1: 0.7442 | Acc: 0.7443 | AUC: 0.8963
  Epoch 3 — Macro F1: 0.7403 | Acc: 0.7391 | AUC: 0.8932
  Epoch 4 — Macro F1: 0.7376 | Acc: 0.7365 | AUC: 0.8920
  Fold 4 best — F1: 0.7442 | Acc: 0.7443 | AUC: 0.8963

--- Fold 5/5 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 — Macro F1: 0.7171 | Acc: 0.7229 | AUC: 0.8954
  Epoch 2 — Macro F1: 0.7587 | Acc: 0.7557 | AUC: 0.9032
  Epoch 3 — Macro F1: 0.7607 | Acc: 0.7589 | AUC: 0.9041
  Epoch 4 — Macro F1: 0.7651 | Acc: 0.7651 | AUC: 0.9039
  Fold 5 best — F1: 0.7651 | Acc: 0.7651 | AUC: 0.9039

===== CV Summary (Optuna params) =====
Mean Macro F1 : 0.7511 +/- 0.0073
Mean Acc      : 0.7512 +/- 0.0074
Mean AUC-ROC  : 0.8966 +/- 0.0038


In [11]:
#Final retraining on full train set + test evaluation
print('Final retrain on full training set...')

full_train_dataset = RedditDataset(train_df['bert_text'].values, train_df['label_enc'].values, tokenizer, MAX_LEN)
test_dataset       = RedditDataset(test_df['bert_text'].values,  test_df['label_enc'].values,  tokenizer, MAX_LEN)
full_train_loader  = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader        = DataLoader(test_dataset,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

final_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

for layer in final_model.distilbert.transformer.layer[:best['num_frozen_layers']]:
    for param in layer.parameters():
        param.requires_grad = False

optimizer   = make_optimizer(final_model, best['lr_base'], best['lr_head'])
total_steps = len(full_train_loader) * FINAL_EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(best['warmup_ratio'] * total_steps),
    num_training_steps=total_steps
)

best_test_f1 = 0.0
best_state   = None

for epoch in range(FINAL_EPOCHS):
    loss = train_epoch(final_model, full_train_loader, optimizer, scheduler, loss_fn)
    f1, acc, auc, preds_tmp, labels_tmp = evaluate(final_model, test_loader)
    print(f'Epoch {epoch+1}/{FINAL_EPOCHS} — Loss: {loss:.4f} | F1: {f1:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f}')
    if f1 > best_test_f1:
        best_test_f1 = f1
        best_state   = copy.deepcopy(final_model.state_dict())
        test_preds   = preds_tmp
        test_labels  = labels_tmp
        test_acc     = acc
        test_auc     = auc

final_model.load_state_dict(best_state)
test_f1 = best_test_f1

print(f'\n===== Test Results (Optuna best params) =====')
print(f'Test Macro F1 : {test_f1:.4f}')
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test AUC-ROC  : {test_auc:.4f}')

Final retrain on full training set...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 — Loss: 0.7879 | F1: 0.7382 | Acc: 0.7425 | AUC: 0.8979
Epoch 2/4 — Loss: 0.5924 | F1: 0.7532 | Acc: 0.7508 | AUC: 0.9055
Epoch 3/4 — Loss: 0.4962 | F1: 0.7619 | Acc: 0.7600 | AUC: 0.9070
Epoch 4/4 — Loss: 0.4365 | F1: 0.7586 | Acc: 0.7588 | AUC: 0.9056

===== Test Results (Optuna best params) =====
Test Macro F1 : 0.7619
Test Accuracy : 0.7600
Test AUC-ROC  : 0.9070


In [12]:
#Classification report + recall 
print(classification_report(test_labels, test_preds, target_names=CLASSES))

from sklearn.metrics import recall_score
print('===== Recall per Class =====')
report = classification_report(test_labels, test_preds, target_names=CLASSES, output_dict=True)
for cls in CLASSES:
    print(f'  {cls:<14} Recall: {report[cls]["recall"]:.4f}')
print(f'\nMacro Recall   : {recall_score(test_labels, test_preds, average="macro",    labels=CLASSES):.4f}')
print(f'Weighted Recall: {recall_score(test_labels, test_preds, average="weighted", labels=CLASSES):.4f}')

              precision    recall  f1-score   support

     Anxiety       0.92      0.86      0.89       800
  Depression       0.66      0.68      0.67       800
SuicideWatch       0.72      0.74      0.73       800

    accuracy                           0.76      2400
   macro avg       0.76      0.76      0.76      2400
weighted avg       0.76      0.76      0.76      2400

===== Recall per Class =====
  Anxiety        Recall: 0.8625
  Depression     Recall: 0.6787
  SuicideWatch   Recall: 0.7388

Macro Recall   : 0.7600
Weighted Recall: 0.7600


In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds, labels=CLASSES)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, cbar=False)
ax.set_title(f'DistilBERT (Optuna) — Macro F1: {test_f1:.3f}', fontsize=11)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_optuna.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {RESULTS_DIR}/confusion_optuna.png')

In [ ]:
# Comparing Optuna vs manual tuning
try:
    manual = pd.read_csv(f'{RESULTS_DIR}/distilbert_results.csv')
    optuna_row = pd.DataFrame([{
        'Model':         'DistilBERT (Optuna)',
        'CV Macro F1':   round(np.mean(cv_f1), 4),
        'CV Std':        round(np.std(cv_f1),  4),
        'Test Macro F1': round(test_f1,        4),
        'Test Acc':      round(test_acc,       4),
        'AUC-ROC':       round(test_auc,       4)
    }])
    comparison = pd.concat([manual, optuna_row], ignore_index=True)
    comparison = comparison.sort_values('Test Macro F1', ascending=False)
    print('===== Manual vs Optuna Comparison =====')
    print(comparison.to_string(index=False))
    comparison.to_csv(f'{RESULTS_DIR}/all_results.csv', index=False)
    print(f'\nSaved {RESULTS_DIR}/all_results.csv')
except FileNotFoundError:
    print(f'distilbert_results.csv not found in {RESULTS_DIR}/ — run the main notebook first.')
    print(f'\nOptuna result: Test Macro F1 = {test_f1:.4f} | Acc = {test_acc:.4f} | AUC = {test_auc:.4f}')